In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import seaborn as sns

In [2]:
# Recreating the clean dataset as created in '03-data-merge-and-cleaning.ipynb'

amazon1 = pd.read_csv('/workspaces/group-project-bas-team/data/AmazonData1.csv')
amazon2 = pd.read_csv('/workspaces/group-project-bas-team/data/AmazonData2.csv')
amazon_data = [amazon1, amazon2]
amazon = pd.concat(amazon_data)

survey = pd.read_csv('/workspaces/group-project-bas-team/data/survey.csv')

data = pd.merge(amazon, survey, on='Survey ResponseID', how='outer')

codes = data[['ASIN/ISBN (Product Code)', 'Title']].drop_duplicates().dropna()
codes_dict = codes.set_index('ASIN/ISBN (Product Code)')['Title'].to_dict()
data['Title'] = data['ASIN/ISBN (Product Code)'].map(codes_dict)

cats = data[['ASIN/ISBN (Product Code)', 'Category']].drop_duplicates().dropna()
cats_dict = cats.set_index('ASIN/ISBN (Product Code)')['Category'].to_dict()
data['Category'] = data['ASIN/ISBN (Product Code)'].map(cats_dict)

data['Q-life-changes'] = data['Q-life-changes'].fillna('None')

data = data.dropna()

data.columns = data.columns.str.replace('Q-', '')\
    .str.replace('demos-', '')\
    .str.replace('amazon-use-', '')\
    .str.replace('substance-use-', '')\
    .str.replace('personal-', '')

data.info()


<class 'pandas.DataFrame'>
Index: 948676 entries, 0 to 1048574
Data columns (total 30 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   Order Date                948676 non-null  str    
 1   Purchase Price Per Unit   948676 non-null  float64
 2   Quantity                  948676 non-null  float64
 3   Shipping Address State    948676 non-null  str    
 4   Title                     948676 non-null  str    
 5   ASIN/ISBN (Product Code)  948676 non-null  str    
 6   Category                  948676 non-null  str    
 7   Survey ResponseID         948676 non-null  str    
 8   age                       948676 non-null  str    
 9   hispanic                  948676 non-null  str    
 10  race                      948676 non-null  str    
 11  education                 948676 non-null  str    
 12  income                    948676 non-null  str    
 13  gender                    948676 non-null  str    
 14  sex

In [3]:
# Exploding columns with multiple values ('life-changes' and 'race')

data['race'] = data['race'].str.split(',')
data['life-changes'] = data['life-changes'].str.split(',')

In [4]:
data = data.explode('race')
data = data.explode('life-changes')

In [5]:
data.info()

<class 'pandas.DataFrame'>
Index: 1095416 entries, 0 to 1048574
Data columns (total 30 columns):
 #   Column                    Non-Null Count    Dtype  
---  ------                    --------------    -----  
 0   Order Date                1095416 non-null  str    
 1   Purchase Price Per Unit   1095416 non-null  float64
 2   Quantity                  1095416 non-null  float64
 3   Shipping Address State    1095416 non-null  str    
 4   Title                     1095416 non-null  str    
 5   ASIN/ISBN (Product Code)  1095416 non-null  str    
 6   Category                  1095416 non-null  str    
 7   Survey ResponseID         1095416 non-null  str    
 8   age                       1095416 non-null  str    
 9   hispanic                  1095416 non-null  str    
 10  race                      1095416 non-null  str    
 11  education                 1095416 non-null  str    
 12  income                    1095416 non-null  str    
 13  gender                    1095416 non-null 

In [6]:
# separating dates into month, day, and year columns
data[['order_month', 'order_day', 'order_year']] = data['Order Date'].str.split('/', expand=True)

data = data.drop('Order Date', axis=1)

data[['order_month', 'order_day', 'order_year']] = data[['order_month', 'order_day', 'order_year']].apply(pd.to_numeric)

data.info()

<class 'pandas.DataFrame'>
Index: 1095416 entries, 0 to 1048574
Data columns (total 32 columns):
 #   Column                    Non-Null Count    Dtype  
---  ------                    --------------    -----  
 0   Purchase Price Per Unit   1095416 non-null  float64
 1   Quantity                  1095416 non-null  float64
 2   Shipping Address State    1095416 non-null  str    
 3   Title                     1095416 non-null  str    
 4   ASIN/ISBN (Product Code)  1095416 non-null  str    
 5   Category                  1095416 non-null  str    
 6   Survey ResponseID         1095416 non-null  str    
 7   age                       1095416 non-null  str    
 8   hispanic                  1095416 non-null  str    
 9   race                      1095416 non-null  str    
 10  education                 1095416 non-null  str    
 11  income                    1095416 non-null  str    
 12  gender                    1095416 non-null  str    
 13  sexual-orientation        1095416 non-null 

In [7]:
# Encoding Categorical Data
# Dropping features not relevant to predicting purchase behavior

cols_to_drop = [
    'Survey ResponseID',
    'sell-YOUR-data',
    'sell-consumer-data',
    'small-biz-use',
    'census-use',
    'research-society'
]

data = data.drop(columns=cols_to_drop)



In [8]:
# Ordinal encoding for columns where the categories have an order
ordinal_mappings = {
    'age': {
        '18 - 24 years': 1,
        '25 - 34 years': 2,
        '35 - 44 years': 3,
        '45 - 54 years': 4,
        '55 - 64 years': 5,
        '65 and older': 6
    },
    'education': {
        'Prefer not to say': 0,
        'Some high school or less': 1,
        'High school diploma or GED': 2,
        "Bachelor's degree": 3,
        'Graduate or professional degree (MA, MS, MBA, PhD, JD, MD, DDS, etc)': 4
    },
    'income': {
        'Prefer not to say': 0,
        'Less than $25,000': 1,
        '$25,000 - $49,999': 2,
        '$50,000 - $74,999': 3,
        '$75,000 - $99,999': 4,
        '$100,000 - $149,999': 5,
        '$150,000 or more': 6
    },
    'howmany': {
        '1 (just me!)': 1,
        '2': 2,
        '3': 3,
        '4+': 4
    },
    'hh-size': {
        '1 (just me!)': 1,
        '2': 2,
        '3': 3,
        '4+': 4
    },
    'how-oft': {
        'Less than 5 times per month': 1,
        '5 - 10 times per month': 2,
        'More than 10 times per month': 3
    }
}

for col, mapping in ordinal_mappings.items():
    data[col] = data[col].map(mapping)
    unmapped = data[col].isna().sum()
    if unmapped > 0:
        print(f"Warning: {unmapped} unmapped values in '{col}' — check category strings match exactly")

print("Ordinal encoding complete.")
data[list(ordinal_mappings.keys())].head()

Ordinal encoding complete.


,age,education,income,howmany,hh-size,how-oft
0,3,3,2,1,1,1
1,3,3,2,1,1,1
3,3,3,2,1,1,1
4,3,3,2,1,1,1
5,3,3,2,1,1,1


In [9]:
# Binary encoding
# hispanic is our only true binary column

yes_no_map = {'Yes': 1, 'No': 0}

data['hispanic'] = data['hispanic'].map(yes_no_map)

In [10]:
# One hot encoding
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False)

data_one_hot = encoder.fit_transform(data[['cigarettes', 'alcohol', 'marijuana', 'diabetes',\
                                            'wheelchair', 'Shipping Address State', 'gender',\
                                            'sexual-orientation', 'state', 'race', 'life-changes']])
df_one_hot = pd.DataFrame(data_one_hot, columns=encoder\
                          .get_feature_names_out(['cigarettes', 'alcohol', 'marijuana', 'diabetes',\
                                                    'wheelchair', 'Shipping Address State', 'gender',\
                                                    'sexual-orientation', 'state', 'race', 'life-changes']))

In [ ]:
data['race'].unique()

<StringArray>
[                       'Black or African American',
                               'White or Caucasian',
 'American Indian/Native American or Alaska Native',
                                            'Asian',
                                            'Other',
        'Native Hawaiian or Other Pacific Islander']
Length: 6, dtype: str

: 

In [ ]:
# Remove one column from each variable to prevent multicollinearity
df_one_hot = df_one_hot.drop(columns=['cigarettes_Prefer not to say', 'alcohol_Prefer not to say', \
                                      'marijuana_Prefer not to say', 'diabetes_Prefer not to say', \
                                        'wheelchair_Prefer not to say', 'Shipping Address State_HI', \
                                        'gender_Prefer not to say', 'sexual-orientation_Prefer not to say', \
                                        'state_I did not reside in the United States', 'race_Other', 'life-changes_None'])

In [ ]:
# Add df_one_hot to existing data and remove original columns

In [ ]:
# No need to encode Title, ASIN/ISBN (Product Code), and category yet, since these will be our potential target values

In [ ]:
# Binary encoding for Yes/No columns

# binary_cols = [
#     'hispanic',
#     'cigarettes',
#     'alcohol',
#     'diabetes',
#     'wheelchair'
# ]

# Preview unique values first to confirm mapping
# for col in binary_cols:
#     print(f"{col}: {data[col].unique()}")

In [ ]:
# Mapping Yes/No to 1/0

# yes_no_map = {'Yes': 1, 'No': 0}

# for col in ['hispanic', 'diabetes', 'wheelchair']:
#     data[col] = data[col].map(yes_no_map)

# Substance use columns have more nuanced values — encode as current user (1) or not (0)
# substance_map = {
#     'Yes': 1,
#     'No': 0,
#     'I stopped in the recent past': 0,
#     'I never did this': 0
# }

# for col in ['cigarettes', 'alcohol', 'marijuana']:
#     data[col] = data[col].map(substance_map)


In [ ]:
# Label encoding for nominal (unordered) categorical columns
# These have no inherent order, so we use integer codes assigned alphabetically

# from sklearn.preprocessing import LabelEncoder

# nominal_cols = [
#     'Shipping Address State',
#     'Title',
#     'ASIN/ISBN (Product Code)',
#     'Category',
#     'gender',
#     'sexual-orientation',
#     'state',
#     'race',
#     'life-changes'
# ]

# le = LabelEncoder()
# label_encoders = {}  # store encoders in case we need to inverse_transform later

# for col in nominal_cols:
#     data[col] = le.fit_transform(data[col].astype(str))
#     label_encoders[col] = le
#     print(f"Encoded '{col}' — {data[col].nunique()} unique values")

In [ ]:
# Final check — confirm all columns are numeric

data.info()
data.describe()
